# Assignment 9 — Image Classification using CNN (Cats vs Dogs)

### **Name:** Manish Satish Payaprapp
### **Reg.No:** 23BCY10046
### **Application.No:** IN26009666

**Objective:** Build a Convolutional Neural Network to classify images of cats and dogs.

**Dataset:** [Dog and Cat Classification Dataset](https://www.kaggle.com/datasets/bhavikjikadara/dog-and-cat-classification-dataset) (Kaggle)



## Task 1: Data Understanding

In [ ]:
# --- Setup: install & authenticate Kaggle, download dataset ---
!pip install -q kaggle tensorflow

import os
from google.colab import files

# Upload your kaggle.json (Kaggle account -> Create New API Token)
if not os.path.exists('/root/.kaggle/kaggle.json'):
    uploaded = files.upload()  # select kaggle.json
    os.makedirs('/root/.kaggle', exist_ok=True)
    for fn in uploaded:
        os.rename(fn, '/root/.kaggle/kaggle.json')
    os.chmod('/root/.kaggle/kaggle.json', 0o600)

!kaggle datasets download -d bhavikjikadara/dog-and-cat-classification-dataset -p /content/data --unzip


In [ ]:
# --- Locate the extracted class folders (Cat/ and Dog/) ---
import os

DATA_ROOT = '/content/data'

def find_class_dirs(root):
    """Walk the extracted dataset and find the two image-class folders."""
    found = {}
    for dirpath, dirnames, filenames in os.walk(root):
        imgs = [f for f in filenames if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
        if imgs:
            name = os.path.basename(dirpath).lower()
            if 'cat' in name:
                found['cat'] = dirpath
            elif 'dog' in name:
                found['dog'] = dirpath
    return found

class_dirs = find_class_dirs(DATA_ROOT)
print('Detected class folders:', class_dirs)
assert 'cat' in class_dirs and 'dog' in class_dirs, "Could not auto-detect Cat/Dog folders — inspect DATA_ROOT manually and set class_dirs by hand."


In [ ]:
# --- Display folder structure ---
def print_tree(root, max_depth=2, prefix=''):
    if max_depth < 0:
        return
    try:
        entries = sorted(os.listdir(root))
    except NotADirectoryError:
        return
    for entry in entries[:10]:  # cap listing for readability
        path = os.path.join(root, entry)
        print(prefix + ('|-- ' if os.path.isdir(path) else '.-- ') + entry)
        if os.path.isdir(path):
            print_tree(path, max_depth - 1, prefix + '    ')

print_tree(DATA_ROOT, max_depth=2)


In [ ]:
# --- Display five sample images with their class labels ---
import matplotlib.pyplot as plt
from PIL import Image

samples = []
for label, folder in class_dirs.items():
    files_in_folder = [f for f in os.listdir(folder) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    for f in files_in_folder[:3]:
        samples.append((os.path.join(folder, f), label))
samples = samples[:5]

fig, axes = plt.subplots(1, len(samples), figsize=(15, 4))
for ax, (path, label) in zip(axes, samples):
    img = Image.open(path)
    ax.imshow(img)
    ax.set_title(label)
    ax.axis('off')
plt.tight_layout()
plt.savefig('sample_images.png', dpi=120)
plt.show()


In [ ]:
# --- Identify: number of classes, image dimensions, total number of images ---
from PIL import Image
import random

num_classes = len(class_dirs)

counts = {}
dims_sample = {}
for label, folder in class_dirs.items():
    files_in_folder = [f for f in os.listdir(folder) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    counts[label] = len(files_in_folder)
    if files_in_folder:
        sample_file = random.choice(files_in_folder)
        with Image.open(os.path.join(folder, sample_file)) as im:
            dims_sample[label] = im.size  # (width, height)

total_images = sum(counts.values())

print(f'Number of classes: {num_classes} -> {list(class_dirs.keys())}')
print(f'Image counts per class: {counts}')
print(f'Sample image dimensions per class (width, height): {dims_sample}')
print(f'Total number of images: {total_images}')
print('Note: raw images have varying dimensions; all will be resized to 128x128 in preprocessing.')


## Task 2: Data Preprocessing

In [ ]:
# --- Rebuild dataset into a clean train/test directory structure Keras can consume ---
# (Some public mirrors of this dataset contain a handful of corrupt/truncated files;
#  we filter those out here so training doesn't crash.)
import shutil
from PIL import Image, UnidentifiedImageError

CLEAN_ROOT = '/content/clean_data'
for split in ['train', 'test']:
    for cls in ['cat', 'dog']:
        os.makedirs(os.path.join(CLEAN_ROOT, split, cls), exist_ok=True)

SPLIT_RATIO = 0.8  # 80% train, 20% test

for label, folder in class_dirs.items():
    files_in_folder = sorted(f for f in os.listdir(folder) if f.lower().endswith(('.jpg', '.jpeg', '.png')))
    valid_files = []
    for f in files_in_folder:
        src = os.path.join(folder, f)
        try:
            with Image.open(src) as im:
                im.verify()
            valid_files.append(f)
        except (UnidentifiedImageError, OSError):
            continue  # skip corrupt file

    split_idx = int(len(valid_files) * SPLIT_RATIO)
    train_files, test_files = valid_files[:split_idx], valid_files[split_idx:]

    for f in train_files:
        shutil.copy(os.path.join(folder, f), os.path.join(CLEAN_ROOT, 'train', label, f))
    for f in test_files:
        shutil.copy(os.path.join(folder, f), os.path.join(CLEAN_ROOT, 'test', label, f))

    print(f'{label}: {len(valid_files)} valid images -> {len(train_files)} train / {len(test_files)} test')


In [ ]:
# --- Resize, normalize (1/255 rescale), and build data generators ---
from tensorflow.keras.preprocessing.image import ImageDataGenerator

IMG_SIZE = (128, 128)
BATCH_SIZE = 32

train_datagen = ImageDataGenerator(rescale=1.0/255)
test_datagen = ImageDataGenerator(rescale=1.0/255)

train_generator = train_datagen.flow_from_directory(
    os.path.join(CLEAN_ROOT, 'train'),
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    shuffle=True,
)

test_generator = test_datagen.flow_from_directory(
    os.path.join(CLEAN_ROOT, 'test'),
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    shuffle=False,
)

print('Class indices:', train_generator.class_indices)


## Task 3: Model Development

In [ ]:
from tensorflow.keras import layers, models

model = models.Sequential([
    layers.Input(shape=(128, 128, 3)),
    layers.Conv2D(32, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(128, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dense(1, activation='sigmoid'),
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.summary()


In [ ]:
EPOCHS = 10

history = model.fit(
    train_generator,
    validation_data=test_generator,
    epochs=EPOCHS,
)


## Task 4: Model Evaluation

In [ ]:
import numpy as np
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay,
)

test_generator.reset()
y_true = test_generator.classes
y_pred_prob = model.predict(test_generator)
y_pred = (y_pred_prob.ravel() > 0.5).astype(int)

test_accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)

print(f'Test Accuracy : {test_accuracy:.4f}')
print(f'Precision     : {precision:.4f}')
print(f'Recall        : {recall:.4f}')
print(f'F1-Score      : {f1:.4f}')


In [ ]:
# --- Confusion Matrix ---
cm = confusion_matrix(y_true, y_pred)
labels = list(train_generator.class_indices.keys())

disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
disp.plot(cmap='Blues')
plt.title('Confusion Matrix')
plt.savefig('confusion_matrix.png', dpi=120, bbox_inches='tight')
plt.show()


In [ ]:
# --- Accuracy vs Epoch and Loss vs Epoch ---
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history.history['accuracy'], label='Train Accuracy')
axes[0].plot(history.history['val_accuracy'], label='Val Accuracy')
axes[0].set_title('Accuracy vs Epoch')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()

axes[1].plot(history.history['loss'], label='Train Loss')
axes[1].plot(history.history['val_loss'], label='Val Loss')
axes[1].set_title('Loss vs Epoch')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()

plt.tight_layout()
plt.savefig('accuracy_loss_vs_epoch.png', dpi=120, bbox_inches='tight')
plt.show()


### Observations

*(Fill this in from your actual run — talking points to look for once you have real numbers:)*

1. **Accuracy trend** — Note whether training accuracy keeps climbing while validation accuracy
   plateaus or dips (a sign of overfitting), or whether both track closely together.
2. **Loss trend** — Check if validation loss starts increasing in later epochs while training
   loss keeps falling — that's the clearest overfitting signal in this architecture.
3. **Precision vs. Recall balance** — Compare which class (cat or dog) the model is more prone
   to misclassifying, visible in the confusion matrix's off-diagonal cells.
4. **Effect of data cleaning** — Mention how many corrupt files were filtered out in
   preprocessing, if any, and whether the final image counts differ meaningfully from the raw
   folder counts you found in Task 1.


## Task 5: Conclusion

*(Replace the bracketed numbers below with your actual results once the notebook has run.)*

This project built a Convolutional Neural Network to classify images of cats and dogs, achieving
a test accuracy of **[X]%** with precision **[X]**, recall **[X]**, and F1-score **[X]**. The three
convolution + max-pooling blocks progressively extracted low-level features (edges, textures) in
early layers and higher-level, more abstract patterns (shapes, fur texture, ear/face structure) in
deeper layers, while pooling reduced spatial dimensions and computational cost without discarding
the most salient activations. This hierarchical feature extraction is the key advantage CNNs hold
over plain ANNs for image tasks: a fully-connected ANN would need a separate weight for every
pixel position and cannot exploit the fact that a "cat ear" looks the same whether it appears in
the top-left or bottom-right of an image, whereas CNN filters share weights and detect patterns
regardless of position. A notable limitation of this approach is that CNNs remain sensitive to
the quality and size of the training set — insufficient or imbalanced data, as well as unusual
poses, lighting, or occlusion not well represented in training, can degrade performance
significantly, and the model requires meaningfully more compute (GPU time) to train than
simpler classical models. Overall, the results confirm CNNs are well suited to this
binary image classification problem, with room for improvement through data augmentation,
transfer learning, or additional regularization to further close the train/validation gap.
